# Home Energy Management — Exploratory Data Analysis

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from src.utils.helpers import load_config
from src.data import HEMSDataLoader, FeatureEngineer

config = load_config()
loader = HEMSDataLoader(config)
df = loader.load()

In [ ]:
report = loader.validate()
print(f"Houses: {report['houses']}")
print(f"Date range: {report['date_range']}")
print(f"Total rows: {report['total_rows']:,}")
df.head()

In [ ]:
df.describe()

## Target Distributions

In [ ]:
targets = list(config['targets'].values())
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, t in zip(axes.flatten(), targets):
    ax.hist(df[t].dropna(), bins=80, edgecolor='black', alpha=0.7)
    ax.set_title(f'{t}')
    ax.set_xlabel('kW')
    ax.set_ylabel('Frequency')
plt.tight_layout()
plt.show()

## Load and PV Over Time (per house)

In [ ]:
houses = df['house_id'].unique()[:3]
fig, axes = plt.subplots(len(houses), 2, figsize=(16, 4*len(houses)))
for i, h in enumerate(houses):
    hh = df[df['house_id'] == h].copy()
    axes[i, 0].plot(hh['timestamp'], hh['total_demand_kW'], linewidth=0.5)
    axes[i, 0].set_title(f'House {h} — Total Demand')
    axes[i, 1].plot(hh['timestamp'], hh['pv_generation_kW'], linewidth=0.5, color='orange')
    axes[i, 1].set_title(f'House {h} — PV Generation')
    for ax in axes[i]:
        ax.set_xlabel('Timestamp')
plt.tight_layout()
plt.show()

## Correlation Analysis

In [ ]:
numeric = df.select_dtypes(include=[np.number]).drop(columns=['house_id'])
corr = numeric.corr()
plt.figure(figsize=(20, 16))
sns.heatmap(corr, cmap='RdBu_r', center=0, vmin=-1, vmax=1, linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.show()

## Key Takeaways
- Multi-house data — model should handle per-house patterns
- PV generation shows clear diurnal pattern with zeros at night
- Load varies by time-of-day and house
- Some features are highly correlated (grid_import ~ total_demand) — consider PCA or feature selection
- Targets at 15-min and 1D horizons have different distributions